# Systemy Big Data: Wykład 2
## Silnik Obliczeniowy Apache Spark (Pod Maska)

Podczas tego dema zobaczymy na żywo koncepcje omawiane na wykładzie:
1. Uruchomimy klaster i zajrzymy do **Spark UI**.
2. Udowodnimy, że Spark jest "leniwy" (Lazy Evaluation).
3. Podejrzymy decyzje optymalizatora **Catalyst** (EXPLAIN).
4. Wywołamy potężne **Wąskie Gardło (Shuffle)** przez Sort-Merge Join.

Najpierw inicjalizujemy nasz lokalny klaster.

In [ ]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
import time

try:
    spark.stop()
except:
    pass

# Uruchamiamy sesję. NADPISUJEMY domyślną konfigurację obrazu Dockera, 
# żeby "zabić" próby łączenia się z nieistniejącym serwerem REST.
spark = SparkSession.builder \
    .appName("SparkUnderTheHoodDemo") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.demo.type", "hadoop") \
    .config("spark.sql.catalog.demo.warehouse", "/tmp") \
    .config("spark.sql.defaultCatalog", "demo") \
    .getOrCreate()

# Uciszamy logi informacyjne w konsoli (chcemy widzieć tylko czyste wyniki)
spark.sparkContext.setLogLevel("ERROR")

print("Klaster Spark Gotowy! (odcięty od sieci!)")

In [ ]:
print("Mój aktywny panel Spark UI działa w kontenerze pod adresem:")
print(spark.sparkContext.uiWebUrl)

### 1. Panel Sterowania Klastrem (Spark UI) 

Kiedy używamy Sparka, zawsze powinniśmy mieć otwarte drugie okno w przeglądarce z panelem administracyjnym (Spark UI). To tam widać, co robią nasze węzły obliczeniowe (Executory).

**[Kliknij tutaj lub otwórz nową kartę: http://localhost:8080](http://localhost:8080)** 

*(Zostaw tę kartę otwartą. Zobaczysz w niej zakładki "Jobs" oraz "Stages", które zapełnią się za chwilę grafami).*

### 2. Generowanie Danych (Duża Tabela)

Stworzymy DataFrame reprezentujący logi z dużego systemu transakcyjnego (5 milionów wierszy).
Użyjemy wbudowanych funkcji Sparka do generowania danych rozproszonych.

In [ ]:
from pyspark.sql.functions import rand, when

print("Generuję plan dla 5 milionów logów...")

# Tworzymy dużą tabelę za pomocą wbudowanego generatora 'range'
df_logs = spark.range(0, 5000000, 1, numPartitions=4) \
    .withColumnRenamed("id", "log_id") \
    .withColumn("user_id", ((rand() * 1000).cast("int") * 10)) \
    .withColumn("status", 
        when(rand() < 0.80, "OK")       # 80% to OK
        .when(rand() < 0.95, "WARN")    # 15% to WARN
        .otherwise("ERROR")             # 5% to ERROR
    )

print("Dane przygotowane (jako obietnica w pamięci Sparka)!")

### 3. Dowód na "Leniwą Ewaluację" (Lazy Evaluation)

Zbudujemy skomplikowany, 3-stopniowy potok (pipeline) obliczeniowy. 
Zmierzymy czas, w jakim Spark to przetworzy.

In [ ]:
start_time = time.time()

# KROK 1: Filtrujemy tylko błędy
df_errors = df_logs.filter(df_logs["status"] == "ERROR")

# KROK 2: Wyciągamy tylko kolumnę z ID użytkownika
df_users_with_errors = df_errors.select("user_id")

# KROK 3: Usuwamy duplikaty
df_unique_users = df_users_with_errors.distinct()

end_time = time.time()
print(f"Czas ZAPLANOWANIA gigantycznych obliczeń: {end_time - start_time:.5f} sekundy")

### 4. Mózg Operacyjny: Catalyst Optimizer

Skoro mamy plan, zapytajmy optymalizatora (Catalysta), jak fizycznie zamierza go wykonać w klastrze. Służy do tego polecenie `explain()`.

In [ ]:
# Tłumaczymy plan z kodu Pythona na fizyczny graf wykonania
df_unique_users.explain()

### 5. Akcja i pojawienie się Shuffle!

Wykonamy naszą "Akcję", każąc Sparkowi policzyć wiersze i wyrzucić wynik na ekran. Wymusi to prawdziwe obliczenia na procesorach.

**Zaraz po wykonaniu tej komórki, wejdź w zakładkę przeglądarki ze Spark UI (localhost:8080) -> Wejdź w zakładkę 'SQL' -> Kliknij w najnowsze zapytanie.**
Zobaczymy słynny Graf DAG i operację sieciową!

In [ ]:
print("Wywołuję AKCJĘ (teraz klastry naprawdę liczą)...")
start_time = time.time()

# Zmuszamy Sparka do pokazania 5 wyników na ekranie
df_unique_users.show(5)

end_time = time.time()
print(f"Rzeczywisty czas OBLICZEŃ (wymuszony przez .show): {end_time - start_time:.2f} sekundy")

### 6. Optymalizacja Złączeń: BHJ vs SMJ

Złączenie dwóch wielkich tabel wywołałoby gigantyczny Shuffle po sieci. Co jeśli jedna tabela (np. Słownik Państw) jest malutka? 

Zrobimy małą tabelę "Kraje" i połączymy z naszymi 5 milionami logów. Wymusimy użycie **Broadcast Hash Join (BHJ)**.

In [ ]:
from pyspark.sql.functions import broadcast

# Tworzymy malutką słownikową tabelę (3 wiersze)
df_countries = spark.createDataFrame([
    (10, "Polska"),
    (20, "Niemcy"),
    (30, "USA")
], ["user_id", "country"])

# Łączymy używając funkcji "broadcast" (nakazujemy wysłać tę małą tabelę do RAMu wszystkich serwerów)
df_joined = df_logs.join(broadcast(df_countries), "user_id")

# Zobaczmy plan fizyczny Catalyst Optimizera!
df_joined.explain()